In [19]:
import os
import json

# -------------------------------------------------------------------
# STEP 1: Ensure directory structure exists
# -------------------------------------------------------------------
os.makedirs("work/notebooks", exist_ok=True)

# -------------------------------------------------------------------
# STEP 2: Define Notebook Cells (Markdown + Code)
# -------------------------------------------------------------------
cells = [
    # --- Cell 1: Header ---
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "# Week 2: ML Task Framing & Problem Formulation\n",
            "**Lane:** Organic Search Rank & Volatility Prediction  \n",
            "**Target File:** `work/notebooks/w02_ml_task_framing.ipynb`\n",
            "\n",
            "This notebook maps FlyRank's rank drop risk estimation task onto the standard Machine Learning lifecycle."
        ]
    },

    # --- Cell 2: Section 1 ---
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "## 1. My Lane as an ML Task\n",
            "\n",
            "* **Task Type:** Binary Classification (predicting whether a page will experience a high-risk position drop in the upcoming temporal window).\n",
            "* **Output:** A calibrated probability score $P(\\text{Drop} \\mid X)$ representing positional volatility risk for a target page."
        ]
    },

    # --- Cell 3: Section 2 ---
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "## 2. Target or Proxy Definition\n",
            "\n",
            "* **Target ($y$):** `is_rank_drop` $\\in \\{0, 1\\}\\n",
            "* **Mathematical Proxy:** A binary label defined as $1$ if `avg_position` drops by more than $3$ positions ($pos_{t+1} - pos_t > 3$) or shifts into positions $> 10$ in the next week ($t+1$), and $0$ otherwise.\n",
            "* **Downstream Action:** Directly flags volatile pages for automated content refresh queues, preventing severe organic traffic loss."
        ]
    },

    # --- Cell 4: Section 3 ---
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "## 3. Success Metric\n",
            "\n",
            "* **Primary Metric:** **PR-AUC (Precision-Recall Area Under Curve)** and **Recall at Top 10% Risk Threshold**.\n",
            "* **Why PR-AUC Over Accuracy/ROC-AUC:** Significant rank drops are inherently rare (class imbalance). High Accuracy is trivialized by predicting zero drops. PR-AUC measures how effectively we flag true drops while minimizing false alarms for editorial teams."
        ]
    },

    # --- Cell 5: Section 4 Header ---
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "## 4. The Unit of Analysis as a Real DataFrame\n",
            "\n",
            "* **Unit of Analysis Definition:** **One Row = One Page per Temporal Snapshot (`page_id` + `week_ending`).**\n",
            "Below, we load the anonymized starter dataset from Hugging Face, format the schema, and construct our clean feature set and target label."
        ]
    },

    # --- Cell 6: Section 4 Code Execution ---
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "import pandas as pd\n",
            "import numpy as np\n",
            "\n",
            "# Load starter dataset\n",
            "url = 'https://huggingface.co/datasets/FlyRank/internship-starter/raw/main/content_refresh_anonymized.csv'\n",
            "df = pd.read_csv(url)\n",
            "df.columns = df.columns.str.lower()\n",
            "\n",
            "# Identify core columns dynamically\n",
            "page_col = 'page_id' if 'page_id' in df.columns else df.columns[0]\n",
            "pos_col = 'avg_position' if 'avg_position' in df.columns else 'position'\n",
            "click_col = 'clicks' if 'clicks' in df.columns else df.columns[1]\n",
            "date_col = 'week_ending' if 'week_ending' in df.columns else 'date'\n",
            "\n",
            "# Ensure chronological sorting per page entity\n",
            "if date_col in df.columns:\n",
            "    df = df.sort_values([page_col, date_col])\n",
            "\n",
            "# 1. Feature Engineering (Strictly Historical t-n Data)\n",
            "df['position_diff'] = df.groupby(page_col)[pos_col].diff()\n",
            "df['click_diff'] = df.groupby(page_col)[click_col].diff()\n",
            "\n",
            "# 2. Target Engineering (Grouped to prevent cross-entity boundary bleeding)\n",
            "df['next_week_pos'] = df.groupby(page_col)[pos_col].shift(-1)\n",
            "df['target_rank_drop'] = ((df['next_week_pos'] - df[pos_col] > 3) | (df['next_week_pos'] > 10)).astype(int)\n",
            "\n",
            "# Drop unobserved tail rows per page entity\n",
            "clean_df = df.dropna(subset=['next_week_pos', 'position_diff']).copy()\n",
            "\n",
            "print('--- Data Frame Schema & Unit of Analysis ---')",
            "print(f'Total Valid Snapshots: {len(clean_df)}')\n",
            "display(clean_df[[page_col, date_col, pos_col, click_col, 'position_diff', 'target_rank_drop']].head(10))"
        ]
    },

    # --- Cell 7: Section 5 ---
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "## 5. Why ML Beats a Fixed Rule Here\n",
            "\n",
            "1. **Non-Linear Interaction Terms:** Fixed rules (e.g., `if position > 10 flag drop`) treat metrics in isolation. ML models capture non-linear interactions between impressions, click-through rates (CTR), and multi-week positional momentum.\n",
            "2. **Algorithm Update Volatility:** Search engine algorithm updates cause noisy, temporary position fluctuations. Fixed heuristics over-trigger on noise. ML classifiers generalize patterns to distinguish true systemic drops from search engine recalculations.\n",
            "3. **Dynamic Prioritization:** Fixed rules yield binary flags without priority ranking. ML probabilistic predictions allow automated sorting so editorial teams prioritize the top 5% highest-risk pages first."
        ]
    },

    # --- Cell 8: Section 6 ---
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "## 6. Self-Check & Submission Audit\n",
            "\n",
            "- [x] **Task Type:** Binary Classification  \n",
            "- [x] **Target/Proxy:** `target_rank_drop` ($pos_{t+1} - pos_t > 3$ or $pos_{t+1} > 10$)  \n",
            "- [x] **Success Metric:** PR-AUC & Top 10% Precision/Recall  \n",
            "- [x] **Unit of Analysis DataFrame:** Verified as `[page_id + week_ending]` snapshot  \n",
            "- [x] **ML vs Heuristic Rationale:** Documented non-linear interactions and noise filtering  \n",
            "- [x] **Content Action:** Direct feed into editorial content refresh priority queue"
        ]
    }
]

# -------------------------------------------------------------------
# STEP 3: Write out valid JSON structure to .ipynb
# -------------------------------------------------------------------
notebook_structure = {
    "cells": cells,
    "metadata": {
        "language_info": {
            "name": "python"
        }
    },
    "nbformat": 4,
    "nbformat_minor": 2
}

file_path = "work/notebooks/w02_ml_task_framing.ipynb"
with open(file_path, "w", encoding="utf-8") as f:
    json.dump(notebook_structure, f, indent=2)

print(f"SUCCESS: Notebook generated at {file_path}")

SUCCESS: Notebook generated at work/notebooks/w02_ml_task_framing.ipynb
